# Hardread TCG — Raw Replay Quality Pipeline (Stages 1–2)

This notebook performs **only** raw replay inventory, provenance recovery, action/schema cleaning, and dataset variant generation. It does not train, evaluate, submit, or download data locally.

Attach the Kaggle daily episode datasets and `kaggle/pokemon-tcg-ai-battle-episodes-index`. All input remains under `/kaggle/input`; all outputs remain under `/kaggle/working/tcg_quality`.


In [ ]:
RUN_STAGE_1 = True
RUN_STAGE_2 = True
MAX_PAIRS = 250_000  # cap each variant to remain within Kaggle output limits
MAX_FILES = 5_000     # statistically meaningful sample; raw data stays on Kaggle
TOP_PCT = 0.30
INPUT_ROOT = "/kaggle/input"
OUT_ROOT = "/kaggle/working/tcg_quality"
print({"stage1": RUN_STAGE_1, "stage2": RUN_STAGE_2, "max_pairs": MAX_PAIRS, "max_files": MAX_FILES, "top_pct": TOP_PCT})


In [ ]:
# Locate attached source and install the quality script in Kaggle working storage.
import os, sys, json, glob, subprocess
# The source dataset contains src/ and its dependencies; this cell avoids any local filesystem assumptions.
SRC = None
for root, dirs, files in os.walk(INPUT_ROOT):
    if "featurizer.py" in files and "replay.py" in files:
        SRC = root
        break
if SRC is None:
    raise FileNotFoundError("Attach hardread-tcg-src (src/featurizer.py and src/replay.py not found)")
sys.path.insert(0, os.path.dirname(SRC)); sys.path.insert(0, SRC)
print("source:", SRC)


In [ ]:
# Embedded code is the version-controlled scripts/kaggle_prepare_quality.py.
QUALITY_SCRIPT = '#!/usr/bin/env python3\n"""Kaggle-only Stage 1/2 raw replay quality pipeline.\n\nInput is attached under /kaggle/input; output must be /kaggle/working. It keeps\nraw replay files out of the repo and preserves provenance in episode_manifest.\n"""\nfrom __future__ import annotations\nimport argparse, collections, glob, json, os, time\nfrom pathlib import Path\nfrom typing import Iterable\nimport numpy as np\nfrom src.replay import parse_episode, featurize_pair, _flush_shard\n\n\ndef files_under(root: str, max_files: int = 0):\n    out=[]\n    for p in glob.glob(os.path.join(root, \'**\', \'*.json\'), recursive=True):\n        try:\n            with open(p) as f: x=json.load(f)\n            if isinstance(x, dict) and isinstance(x.get(\'steps\'), list) and isinstance(x.get(\'info\'), dict): out.append(p)\n        except Exception: pass\n        if max_files and len(out) >= max_files: break\n    return sorted(out)\n\ndef _reward_value(x):\n    return float(x) if isinstance(x,(int,float,np.integer,np.floating)) else 0.0\n\ndef scan(files):\n    rows=[]; stats=collections.Counter(); agent=collections.defaultdict(collections.Counter)\n    for i,p in enumerate(files):\n        ep=parse_episode(p)\n        if ep is None: stats[\'parse_errors\']+=1; continue\n        info=ep.get(\'info\') or {}; names=(info.get(\'TeamNames\') or [])[:2]; rewards=ep.get(\'rewards\') or [0,0]\n        status=ep.get(\'statuses\') or []\n        winner=-1\n        if len(rewards)>=2:\n            r0, r1 = _reward_value(rewards[0]), _reward_value(rewards[1])\n            winner=0 if r0>r1 else 1 if r1>r0 else -1\n        for pi,name in enumerate(names):\n            if not name: continue\n            if winner<0: agent[name][\'draws\']+=1\n            elif pi==winner: agent[name][\'wins\']+=1\n            else: agent[name][\'losses\']+=1\n        row={\'episode_id\':info.get(\'EpisodeId\',i),\'source_file\':p,\'source_dataset\':next((x for x in p.split(\'/\') if \'episodes-\' in x),None),\'team_names\':names,\'rewards\':rewards[:2],\'winner\':winner,\'statuses\':status[:2],\'n_steps\':len(ep.get(\'steps\') or []),\'file_size\':os.path.getsize(p)}\n        rows.append(row); stats[\'episodes\']+=1; stats[f\'winner_{winner}\']+=1\n    return rows,agent,stats\n\ndef rates(agent):\n    out={}\n    for n,s in agent.items():\n        games=s[\'wins\']+s[\'losses\']; out[n]={\'wins\':s[\'wins\'],\'losses\':s[\'losses\'],\'draws\':s[\'draws\'],\'games\':games,\'win_rate\':s[\'wins\']/games if games else 0.0}\n    return out\n\ndef clean_pairs(ep, allowed=None, deck_match=True):\n    """Extract only normal option-index actions; exclude deck selection and malformed labels."""\n    info=ep.get("info") or {}; names=(info.get("TeamNames") or [])[:2]\n    rewards=ep.get("rewards") or [0, 0]; winner=-1\n    if len(rewards)>=2:\n        r0, r1 = _reward_value(rewards[0]), _reward_value(rewards[1])\n        winner=0 if r0>r1 else 1 if r1>r0 else -1\n    players={i for i,n in enumerate(names) if allowed is None or n in allowed}\n    if not players: return []\n    if deck_match:\n        # Reuse source parser\'s deck detector by extracting per-player visible IDs.\n        from src.replay import _collect_player_card_ids, match_deck\n        dids={i:match_deck(_collect_player_card_ids(ep.get("steps") or [],i)) for i in (0,1)}\n    else: dids={0:0,1:0}\n    out=[]\n    for step in ep.get("steps") or []:\n        if not isinstance(step,list): continue\n        for pi,entry in enumerate(step[:2]):\n            if pi not in players or not isinstance(entry,dict): continue\n            obs=entry.get("observation"); action=entry.get("action")\n            sel=(obs or {}).get("select") if isinstance(obs,dict) else None\n            if not isinstance(sel,dict) or not sel.get("option") or not isinstance(action,list) or not action: continue\n            # type 9 and long card-ID lists are deck-selection, not policy actions.\n            if int(sel.get("type",-1))==9 or len(action)>=30: continue\n            opts=sel.get("option") or []\n            if any(not isinstance(i,(int,np.integer)) or not 0<=int(i)<len(opts) for i in action): continue\n            if len(set(int(i) for i in action))>int(sel.get("maxCount",len(action))): continue\n            if winner<0: outcome=0.0\n            else: outcome=1.0 if pi==winner else -1.0\n            out.append((obs,[int(i) for i in action],pi,outcome,int((info.get("EpisodeId") or 0)),dids[pi]))\n    return out\n\ndef build_pairs(files, allowed, out_dir, deck_match=True, max_pairs=0, shard_size=5000):\n    Path(out_dir).mkdir(parents=True,exist_ok=True); shard=[]; total=0; rejected=collections.Counter(); deck=collections.Counter(); by_ep=collections.Counter(); idx=0\n    for fp in files:\n        ep=parse_episode(fp)\n        if ep is None: continue\n        info=ep.get(\'info\') or {}; names=(info.get(\'TeamNames\') or [])[:2]; eid=info.get(\'EpisodeId\',idx)\n        # extract_pairs applies deck-selection and action-count validation.\n        pairs=clean_pairs(ep,allowed=allowed,deck_match=deck_match)\n        if pairs:\n            by_ep[int(eid)]+=len(pairs)\n        for obs,action,pidx,outcome,eid2,did in pairs:\n            rec=featurize_pair(obs,action,deck_id=did,outcome=outcome,episode_id=eid2)\n            if rec is None: rejected[\'featurize\']+=1; continue\n            shard.append(rec); deck[int(did)]+=1; total+=1\n            if max_pairs and total>=max_pairs: break\n            if len(shard)>=shard_size:\n                _flush_shard(shard,out_dir,idx); idx+=1; shard=[]\n        if max_pairs and total>=max_pairs: break\n    if shard: _flush_shard(shard,out_dir,idx); idx+=1\n    return {\'pairs\':total,\'shards\':idx,\'deck_pairs\':dict(deck),\'episodes_with_pairs\':len(by_ep),\'rejected\':dict(rejected)}\n\ndef audit_shards(out_dir):\n    """Verify generated labels obey submission-policy action-count invariants."""\n    files=sorted(glob.glob(os.path.join(out_dir, \'shard_*.npz\')))\n    n=viol=0\n    for fp in files:\n        z=np.load(fp); chosen=(z[\'chosen\']>0).sum(axis=1); max_count=z[\'max_count\']\n        n += len(chosen); viol += int((chosen > max_count).sum())\n    return {\'pairs\':n, \'shards\':len(files), \'chosen_gt_max_count\':viol}\n\ndef main():\n    ap=argparse.ArgumentParser(); ap.add_argument(\'--input-root\',default=\'/kaggle/input\'); ap.add_argument(\'--out\',default=\'/kaggle/working/quality\'); ap.add_argument(\'--top-pct\',type=float,default=.3); ap.add_argument(\'--max-pairs\',type=int,default=0); ap.add_argument(\'--max-files\',type=int,default=0); args=ap.parse_args()\n    out=Path(args.out); out.mkdir(parents=True,exist_ok=True); files=files_under(args.input_root, args.max_files); print(\'raw files\',len(files))\n    rows,agent,scan_stats=scan(files); ar=rates(agent); ranked=sorted(ar,key=lambda n:(ar[n][\'win_rate\'],ar[n][\'games\']),reverse=True)\n    min_games=10; eligible=[n for n in ranked if ar[n][\'games\']>=min_games]; cut=max(1,int(len(eligible)*args.top_pct)); top=set(eligible[:cut])\n    for row in rows:\n        row[\'agent_stats\']=[ar.get(n,{}) for n in row[\'team_names\']]\n        row[\'quality_tier\']=\'top_empirical_winrate\' if any(n in top for n in row[\'team_names\']) else \'all_valid\'\n    with open(out/\'episode_manifest.jsonl\',\'w\') as f:\n        for r in rows: f.write(json.dumps(r,sort_keys=True)+\'\\n\')\n    summary={\'source_root\':os.path.abspath(args.input_root),\'created_at\':time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()),\'raw_files\':len(files),\'scan\':dict(scan_stats),\'agents\':ar,\'eligible_agents\':len(eligible),\'top_agents\':sorted(top),\'top_pct\':args.top_pct}\n    (out/\'quality_summary.json\').write_text(json.dumps(summary,indent=2)+\'\\n\')\n    variants={}\n    variants[\'all_valid\']=build_pairs(files,None,str(out/\'pairs_all\'),max_pairs=args.max_pairs)\n    variants[\'top_empirical_winrate\']=build_pairs(files,top,str(out/\'pairs_top\'),max_pairs=args.max_pairs)\n    variants[\'integrity\']={\'all_valid\':audit_shards(str(out/\'pairs_all\')), \'top_empirical_winrate\':audit_shards(str(out/\'pairs_top\'))}\n    (out/\'variant_summary.json\').write_text(json.dumps(variants,indent=2)+\'\\n\')\n    print(json.dumps({\'summary\':summary|{\'variants\':variants}},indent=2)[:12000])\nif __name__==\'__main__\': main()\n'
script_path = "/kaggle/working/kaggle_prepare_quality.py"
with open(script_path, "w") as f: f.write(QUALITY_SCRIPT)
print("quality script:", script_path)


## Stage 1 — metadata and schema quality

The script scans all attached raw episodes, preserves source file/dataset, agent names, team names, rewards, statuses, and episode IDs, and computes empirical agent win rates. Raw replay data is never copied into the repository.


In [ ]:
if RUN_STAGE_1:
    import runpy, sys
    sys.argv = ["kaggle_prepare_quality.py", "--input-root", INPUT_ROOT, "--out", OUT_ROOT, "--top-pct", str(TOP_PCT), "--max-pairs", str(MAX_PAIRS), "--max-files", str(MAX_FILES)]
    # Stage 1 scan plus Stage 2 pair generation are run together only after flags are checked.
    runpy.run_path(script_path, run_name="__main__")
else:
    print("RUN_STAGE_1=False; skipping raw scan")


## Stage 2 — quality variants

Outputs:

- `episode_manifest.jsonl`: provenance and agent/opponent metadata per episode.
- `quality_summary.json`: global counts and empirical agent quality.
- `variant_summary.json`: pair counts and deck distributions.
- `pairs_all/`: cleaned baseline pairs.
- `pairs_top/`: pairs where at least one player is in the top empirical win-rate tier.

Deck balancing is applied later through manifest weights; no deck or unknown class is silently deleted.


In [ ]:
import os, json, glob
if os.path.exists(OUT_ROOT):
    print("quality output exists; refusing to overwrite:", OUT_ROOT)
else:
    print("Outputs are written by the Stage 1/2 run above.")
print("artifacts:", sorted(glob.glob(OUT_ROOT + "/**", recursive=True))[:20])
